Daniel Iván Lozano Simanca

Samuel Ochoa Alzate

In [9]:
import scipy.stats as stats
from scipy.signal import find_peaks
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

def energia_promedio(signal):
    datos = signal['data']

    energia_canal = np.sum(datos ** 2, axis=1)
    promedio_canal = np.mean(energia_canal, axis=1)

    # for i, e in enumerate(promedio_canal):
    #     print(f"Canal {i+1}: {e:.4f}")

    return promedio_canal

In [ ]:
#2
def dataframe(carpeta):
    filas = {}
    
    for archivo in os.listdir(carpeta):
        if archivo.endswith('.mat'):
            sujeto = archivo.replace('_EP_reposo.mat', '')
            ruta = os.path.join(carpeta, archivo)
            
            signal = sio.loadmat(ruta)
            energia = energia_promedio(signal)
            
            filas[sujeto] = energia
    
    df = pd.DataFrame(filas, index=[f'Canal_{i+1}' for i in range(8)]).T
    df.index.name = '#sujeto'
    return df

df_parkinson = dataframe('parkinson')
df_control   = dataframe('control')

print(df_parkinson)
print(df_control)

              Canal_1       Canal_2       Canal_3       Canal_4       Canal_5  \
#sujeto                                                                         
P001     12438.243570  11261.175800  10819.634775   9489.784462  12091.060945   
P004     17995.660058  12001.601821  12286.344400  14785.908284  17058.433161   
P005     38092.102574  43575.379457  41979.994799  41715.287990  46513.737045   
P007     23742.325612  22070.007569  24540.315612  21803.936448  22594.339745   
P012     48574.518921  51806.529769  73171.952374  59707.699631  56552.175747   
P013     16202.416566  13124.247855  13988.674335  12752.027365  15784.724049   
P015     10692.948223  10841.187262  12154.390086  24161.685202  14789.173543   
P016     12157.229828  13398.658526  17668.877657  14841.104693  11297.742247   
P017      9581.810471  14008.572615   9589.230257   9374.085669   8154.941858   
P018     23658.738825  23990.255991  30633.745996  22888.894132  19932.315538   
P020     23446.051598  26091

In [ ]:
#3
pVals_normalidad = {}

for canal in df_parkinson.columns:
    _, p_park = stats.shapiro(df_parkinson[canal])
    _, p_ctrl = stats.shapiro(df_control[canal])
    
    pVals_normalidad[canal] = {
        'p_parkinson': p_park,
        'p_control'  : p_ctrl,
        'Normalidad en ambos grupos': (p_park > 0.05) and (p_ctrl > 0.05)
    }

df_normalidad = pd.DataFrame(pVals_normalidad).T
print(df_normalidad)

        p_parkinson p_control Normalidad en ambos grupos
Canal_1    0.014392  0.006253                      False
Canal_2    0.004522  0.003954                      False
Canal_3     0.00091  0.008902                      False
Canal_4    0.000312  0.000197                      False
Canal_5    0.005323    0.0008                      False
Canal_6     0.00001  0.000006                      False
Canal_7    0.000017  0.000004                      False
Canal_8    0.000009  0.000001                      False


Como por lo menos en uno de los canales no se pudo confirmar el supuesto de normalidad (en realidad en ninguno se dió), no tiene sentido seguir con la comprobación de los demás requisitos, pues ya no se puede aplicar t test.

Por consecuente, se aplica la prueba U de Mann-Whitney no paramétrica para determinar la significancia estadística de la diferencia entre estos grupos.

In [14]:
pVals_final = {}

for canal in df_parkinson.columns:
    grupo_park = df_parkinson[canal]
    grupo_ctrl = df_control[canal]

    _, p_val = stats.mannwhitneyu(grupo_park, grupo_ctrl, alternative='two-sided')
    pVals_final[canal] = {
        'prueba': 'Mann-Whitney',
        'p_valor': round(p_val, 4),
        'diferencia_significativa': p_val < 0.05
        }

df_final = pd.DataFrame(pVals_final).T
print(df_final)

               prueba p_valor diferencia_significativa
Canal_1  Mann-Whitney  0.4057                    False
Canal_2  Mann-Whitney  0.5705                    False
Canal_3  Mann-Whitney  0.4604                    False
Canal_4  Mann-Whitney  0.2345                    False
Canal_5  Mann-Whitney    0.56                    False
Canal_6  Mann-Whitney  0.2801                    False
Canal_7  Mann-Whitney  0.1183                    False
Canal_8  Mann-Whitney  0.1505                    False


En base a la prueba U, no se encontraron significativas las diferencias en cuanto a las medias de los grupos parkinson y control para el caso de ningún canal. Esto nos quiere decir que la energía promedio por canal no es suficiente para distinguir entre pacientes con Parkinson y control sanos en estado de reposo, y hay razones tanto estadísticas como fisiológicas que lo explican.

Por un lado, Shapiro-Wilk rechazó normalidad en todos los canales, los que sugiere los datos no siguen una distribución normal, lo cual es común en señales EEG. Por otro, La alta variabilidad a nivel interno de cada grupo hace que las distribuciones de ambos grupos se solapen completamente, imposibilitando la separación, puesto que se podían apreciar varianzas considerablemente altas para ambos grupos en los canales 6 a 8.